# Feature Store 与数据版本控制教程

> **前置知识**: Python基础、机器学习特征工程概念
>
> **学习目标**: 掌握特征存储的设计原理和数据血缘追踪方法

---

## 为什么需要 Feature Store？

```
传统特征管理的问题:
┌─────────────────────────────────────────────────────────────┐
│  "训练和推理用的特征计算逻辑不一致"   → 训练-推理偏差     │
│  "这个特征是怎么计算出来的？"         → 无法追溯          │
│  "多个团队重复计算相同的特征"         → 资源浪费          │
│  "线上推理需要低延迟获取特征"         → 性能瓶颈          │
└─────────────────────────────────────────────────────────────┘

Feature Store 解决方案:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐     │
│  │ 特征定义    │ →  │ 特征计算    │ →  │ 特征存储    │     │
│  │ (Schema)    │    │ (Pipeline)  │    │ (Store)     │     │
│  └─────────────┘    └─────────────┘    └─────────────┘     │
│                                              │              │
│                           ┌──────────────────┴───────────┐ │
│                           ▼                              ▼ │
│                    ┌─────────────┐              ┌─────────────┐│
│                    │ 离线存储    │              │ 在线存储    ││
│                    │ (训练)      │              │ (推理)      ││
│                    │ 高吞吐      │              │ 低延迟      ││
│                    └─────────────┘              └─────────────┘│
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **Feature Store 基础** - 特征定义、在线/离线存储
2. **数据血缘追踪** - 追踪特征的来源和转换历史
3. **特征工程流水线** - 自动化特征处理

In [ ]:
# ============================================================
# 环境准备
# ============================================================
# 标准库
import numpy as np
import time
import hashlib      # 用于数据校验和
import json
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional
from datetime import datetime

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"NumPy 版本: {np.__version__}")

## 1. Feature Store 基础

**核心概念**: Feature Store 是集中管理和服务机器学习特征的系统

```
Feature Store 核心组件:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  Feature (特征定义)                                         │
│  ├── name: 特征名称                                        │
│  ├── dtype: 数据类型 (int, float, string)                  │
│  ├── entity: 实体类型 (user, item, order)                  │
│  ├── description: 特征描述                                 │
│  └── ttl_seconds: 过期时间                                 │
│                                                             │
│  在线存储 (Online Store)                                    │
│  ├── 用途: 实时推理                                        │
│  ├── 特点: 低延迟 (<10ms)、高可用                          │
│  └── 存储: Redis、DynamoDB 等 KV 存储                      │
│                                                             │
│  离线存储 (Offline Store)                                   │
│  ├── 用途: 模型训练                                        │
│  ├── 特点: 高吞吐、支持时间旅行查询                        │
│  └── 存储: Hive、S3、BigQuery 等数据湖                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘

在线 vs 离线存储对比:
┌─────────────────────────────────────────────────────────────┐
│  特性        在线存储              离线存储                 │
│  ────────    ────────              ────────                 │
│  延迟        <10ms                 秒~分钟级                │
│  吞吐量      中等                  高                       │
│  数据量      最新值                历史全量                 │
│  用途        实时推理              批量训练                 │
│  存储成本    高                    低                       │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Feature Store 核心实现
# ============================================================

@dataclass
class Feature:
    """
    特征定义
    
    描述一个特征的元数据，包括名称、类型、所属实体等
    
    参数说明:
    ┌─────────────────────────────────────────────────────────┐
    │  name: 特征名称，如 "user_age", "item_price"           │
    │  dtype: 数据类型，如 "int", "float", "string"          │
    │  description: 特征描述，说明特征含义                   │
    │  entity: 实体类型，如 "user", "item", "order"          │
    │  online: 是否支持在线获取                              │
    │  offline: 是否支持离线获取                             │
    │  ttl_seconds: 过期时间（秒），默认 24 小时             │
    └─────────────────────────────────────────────────────────┘
    """
    name: str                    # 特征名称
    dtype: str                   # 数据类型
    description: str             # 特征描述
    entity: str                  # 实体类型
    online: bool = True          # 是否支持在线获取
    offline: bool = True         # 是否支持离线获取
    ttl_seconds: int = 86400     # 过期时间（秒）


class FeatureStore:
    """
    特征存储
    
    核心功能:
    1. register_feature(): 注册特征定义
    2. ingest(): 写入特征值（同时写入在线和离线存储）
    3. get_online_features(): 在线获取（低延迟，用于推理）
    4. get_historical_features(): 离线获取（批量，用于训练）
    
    存储架构:
    ┌─────────────────────────────────────────────────────────┐
    │  在线存储 (online_store)                               │
    │  ├── 结构: {entity_id: {feature_name: value}}         │
    │  ├── 特点: 只存储最新值，低延迟访问                   │
    │  └── 用途: 实时推理                                   │
    │                                                         │
    │  离线存储 (offline_store)                              │
    │  ├── 结构: [{entity_id, timestamp, features...}]      │
    │  ├── 特点: 存储历史全量，支持时间范围查询             │
    │  └── 用途: 模型训练、特征回填                         │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self):
        self.features: Dict[str, Feature] = {}      # 特征定义注册表
        self.online_store: Dict[str, Dict] = {}     # 在线存储: entity_id -> features
        self.offline_store: List[Dict] = []         # 离线存储: 历史记录列表
    
    def register_feature(self, feature: Feature):
        """
        注册特征定义
        
        在使用特征之前，需要先注册其元数据
        """
        self.features[feature.name] = feature
        print(f"注册特征: {feature.name} ({feature.dtype}, entity={feature.entity})")
    
    def ingest(self, entity_id: str, features: Dict[str, Any], timestamp: float = None):
        """
        写入特征值
        
        同时写入在线存储和离线存储，确保训练和推理使用相同的数据
        
        参数:
            entity_id: 实体ID，如 "user_123"
            features: 特征字典，如 {"user_age": 25, "user_spend": 100.0}
            timestamp: 时间戳，默认当前时间
        """
        timestamp = timestamp or time.time()
        
        # 写入在线存储（覆盖旧值）
        if entity_id not in self.online_store:
            self.online_store[entity_id] = {}
        self.online_store[entity_id].update(features)
        self.online_store[entity_id]['_timestamp'] = timestamp
        
        # 写入离线存储（追加记录）
        record = {'entity_id': entity_id, 'timestamp': timestamp, **features}
        self.offline_store.append(record)
    
    def get_online_features(self, entity_id: str, feature_names: List[str]) -> Optional[Dict]:
        """
        在线获取特征（低延迟）
        
        用于实时推理，只返回最新值
        
        参数:
            entity_id: 实体ID
            feature_names: 需要获取的特征名称列表
            
        返回:
            特征字典，如 {"user_age": 25, "user_spend": 100.0}
        """
        if entity_id not in self.online_store:
            return None
        return {k: self.online_store[entity_id].get(k) for k in feature_names}
    
    def get_historical_features(
        self,
        entity_ids: List[str],
        feature_names: List[str],
        start_time: float = None,
        end_time: float = None
    ) -> List[Dict]:
        """
        离线获取历史特征（批量）
        
        用于模型训练，支持时间范围过滤
        
        参数:
            entity_ids: 实体ID列表
            feature_names: 特征名称列表
            start_time: 开始时间（可选）
            end_time: 结束时间（可选）
            
        返回:
            历史特征记录列表
        """
        results = []
        for record in self.offline_store:
            # 过滤实体
            if record['entity_id'] not in entity_ids:
                continue
            # 过滤时间范围
            if start_time and record['timestamp'] < start_time:
                continue
            if end_time and record['timestamp'] > end_time:
                continue
            # 提取指定特征
            results.append({
                k: record.get(k) 
                for k in ['entity_id', 'timestamp'] + feature_names
            })
        return results

In [ ]:
# ============================================================
# Feature Store 实战演示
# ============================================================
print("=" * 60)
print("Feature Store 实战演示")
print("=" * 60)

# 创建 Feature Store 实例
store = FeatureStore()

# ============================================================
# 步骤1: 注册特征定义
# ============================================================
print("\n步骤1: 注册特征定义")
print("-" * 40)

store.register_feature(Feature(
    name='user_age',
    dtype='int',
    description='用户年龄',
    entity='user'
))

store.register_feature(Feature(
    name='user_spend',
    dtype='float',
    description='用户累计消费金额',
    entity='user'
))

store.register_feature(Feature(
    name='user_visits',
    dtype='int',
    description='用户访问次数',
    entity='user'
))

# ============================================================
# 步骤2: 写入特征数据
# ============================================================
print("\n步骤2: 写入特征数据")
print("-" * 40)

for i in range(5):
    entity_id = f'user_{i}'
    features = {
        'user_age': 20 + i * 5,
        'user_spend': 100.0 * (i + 1),
        'user_visits': 10 * (i + 1)
    }
    store.ingest(entity_id, features)
    print(f"  写入 {entity_id}: age={features['user_age']}, spend={features['user_spend']}")

# ============================================================
# 步骤3: 在线查询（实时推理场景）
# ============================================================
print("\n步骤3: 在线查询（实时推理）")
print("-" * 40)

online_features = store.get_online_features('user_2', ['user_age', 'user_spend'])
print(f"  user_2 的在线特征: {online_features}")

# ============================================================
# 步骤4: 离线查询（模型训练场景）
# ============================================================
print("\n步骤4: 离线查询（模型训练）")
print("-" * 40)

historical_features = store.get_historical_features(
    entity_ids=['user_0', 'user_1'],
    feature_names=['user_age', 'user_spend']
)
print(f"  历史特征记录数: {len(historical_features)}")
for record in historical_features:
    print(f"    {record['entity_id']}: age={record['user_age']}, spend={record['user_spend']}")

## 2. 数据血缘追踪

**核心概念**: 数据血缘（Data Lineage）追踪数据从源头到最终使用的完整路径

```
数据血缘的价值:
┌─────────────────────────────────────────────────────────────┐
│  "这个特征是怎么计算出来的？"         → 可追溯            │
│  "上游数据变了，哪些模型会受影响？"   → 影响分析          │
│  "模型预测错误，问题出在哪里？"       → 根因定位          │
│  "如何复现三个月前的训练数据？"       → 数据版本控制      │
└─────────────────────────────────────────────────────────────┘

血缘追踪数据结构:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  DataLineage (数据血缘记录)                                 │
│  ├── dataset_id: 数据集唯一标识                            │
│  ├── version: 版本号                                       │
│  ├── source: 数据来源 (database, file, transformation)     │
│  ├── transformations: 应用的转换列表                       │
│  ├── parent_datasets: 父数据集ID列表                       │
│  ├── created_at: 创建时间                                  │
│  └── checksum: 数据校验和（用于验证数据完整性）            │
│                                                             │
│  血缘链示例:                                                │
│  raw_data → normalized_data → selected_features → model    │
│     │              │                │                       │
│     ▼              ▼                ▼                       │
│  database      zscore标准化      特征选择                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 数据血缘追踪实现
# ============================================================

@dataclass
class DataLineage:
    """
    数据血缘记录
    
    记录数据集的来源、转换历史和版本信息
    
    字段说明:
    ┌─────────────────────────────────────────────────────────┐
    │  dataset_id: 数据集唯一标识                            │
    │  version: 版本号                                       │
    │  source: 数据来源 (database, file, transformation)     │
    │  transformations: 应用的转换操作列表                   │
    │  parent_datasets: 父数据集ID（用于追溯上游）           │
    │  created_at: 创建时间                                  │
    │  checksum: 数据校验和（验证数据完整性）                │
    └─────────────────────────────────────────────────────────┘
    """
    dataset_id: str                                              # 数据集ID
    version: str                                                 # 版本号
    source: str                                                  # 数据来源
    transformations: List[Dict] = field(default_factory=list)   # 转换操作
    parent_datasets: List[str] = field(default_factory=list)    # 父数据集
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    checksum: str = ""                                           # 数据校验和


class DataLineageTracker:
    """
    数据血缘追踪器
    
    核心功能:
    1. register_dataset(): 注册原始数据集
    2. record_transformation(): 记录数据转换
    3. get_full_lineage(): 获取完整血缘链
    
    血缘追踪原理:
    ┌─────────────────────────────────────────────────────────┐
    │  每次数据转换都会创建新的血缘记录:                     │
    │                                                         │
    │  raw_data (source=database)                            │
    │      ↓ normalize                                       │
    │  normalized_data (parent=raw_data)                     │
    │      ↓ feature_select                                  │
    │  selected_features (parent=normalized_data)            │
    │                                                         │
    │  通过 parent_datasets 可以追溯完整的数据来源链         │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self):
        self.lineage_records: Dict[str, DataLineage] = {}
    
    def register_dataset(self, dataset_id: str, source: str, data) -> DataLineage:
        """
        注册原始数据集
        
        参数:
            dataset_id: 数据集唯一标识
            source: 数据来源（如 database, file, api）
            data: 数据内容（用于计算校验和）
            
        返回:
            创建的血缘记录
        """
        # 计算数据校验和（用于验证数据完整性）
        checksum = hashlib.md5(str(data).encode()).hexdigest()[:8]
        
        lineage = DataLineage(
            dataset_id=dataset_id,
            version="1.0",
            source=source,
            checksum=checksum
        )
        self.lineage_records[dataset_id] = lineage
        return lineage
    
    def record_transformation(
        self,
        input_id: str,
        transform_name: str,
        params: Dict,
        output_id: str
    ) -> Optional[DataLineage]:
        """
        记录数据转换
        
        参数:
            input_id: 输入数据集ID
            transform_name: 转换名称（如 normalize, feature_select）
            params: 转换参数
            output_id: 输出数据集ID
            
        返回:
            创建的血缘记录
        """
        parent = self.lineage_records.get(input_id)
        if parent:
            new_lineage = DataLineage(
                dataset_id=output_id,
                version="1.0",
                source="transformation",
                transformations=[{"name": transform_name, "params": params}],
                parent_datasets=[input_id]
            )
            self.lineage_records[output_id] = new_lineage
            return new_lineage
        return None
    
    def get_full_lineage(self, dataset_id: str) -> List[DataLineage]:
        """
        获取完整血缘链
        
        从指定数据集向上追溯，返回完整的数据来源链
        
        参数:
            dataset_id: 数据集ID
            
        返回:
            血缘记录列表（从当前到源头）
        """
        lineage = []
        current = self.lineage_records.get(dataset_id)
        
        while current:
            lineage.append(current)
            # 追溯父数据集
            if current.parent_datasets:
                current = self.lineage_records.get(current.parent_datasets[0])
            else:
                break
        
        return lineage


# ============================================================
# 数据血缘追踪演示
# ============================================================
print("=" * 60)
print("数据血缘追踪演示")
print("=" * 60)

# 创建血缘追踪器
tracker = DataLineageTracker()

# 步骤1: 注册原始数据
print("\n步骤1: 注册原始数据")
print("-" * 40)
raw_data = np.random.randn(100, 10)
tracker.register_dataset('raw_data', 'database', raw_data)
print(f"  注册数据集: raw_data (来源: database)")

# 步骤2: 记录数据转换
print("\n步骤2: 记录数据转换")
print("-" * 40)

# 转换1: 标准化
tracker.record_transformation(
    input_id='raw_data',
    transform_name='normalize',
    params={'method': 'zscore'},
    output_id='normalized_data'
)
print(f"  raw_data → normalize → normalized_data")

# 转换2: 特征选择
tracker.record_transformation(
    input_id='normalized_data',
    transform_name='feature_select',
    params={'k': 5},
    output_id='selected_features'
)
print(f"  normalized_data → feature_select → selected_features")

# 步骤3: 查看完整血缘
print("\n步骤3: 查看完整血缘链")
print("-" * 40)
lineage = tracker.get_full_lineage('selected_features')
for i, record in enumerate(lineage):
    indent = "  " * i
    parent_info = f" ← {record.parent_datasets}" if record.parent_datasets else " (源头)"
    print(f"  {indent}{record.dataset_id}{parent_info}")

## 3. 特征工程流水线

**核心概念**: 将特征计算、血缘追踪整合成自动化流水线

```
特征工程流水线架构:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  输入数据 → 转换器1 → 转换器2 → ... → 输出特征            │
│     │          │          │              │                  │
│     ▼          ▼          ▼              ▼                  │
│  注册血缘   记录转换   记录转换      存入Feature Store     │
│                                                             │
└─────────────────────────────────────────────────────────────┘

流水线组件:
┌─────────────────────────────────────────────────────────────┐
│  FeaturePipeline (特征流水线)                               │
│  ├── feature_store: 特征存储实例                           │
│  ├── lineage_tracker: 血缘追踪器                           │
│  └── transformers: 转换器列表                              │
│                                                             │
│  Transformer (转换器)                                       │
│  ├── name: 转换名称                                        │
│  ├── fn: 转换函数                                          │
│  └── params: 转换参数                                      │
└─────────────────────────────────────────────────────────────┘

常用特征转换:
┌─────────────────────────────────────────────────────────────┐
│  转换类型        说明                  示例                 │
│  ──────────      ────                  ────                 │
│  标准化          缩放到标准分布        zscore, minmax       │
│  离散化          连续值转离散          分桶, 等频           │
│  编码            类别转数值            one-hot, label       │
│  聚合            多行聚合为一行        sum, mean, count     │
│  交叉            特征组合              A*B, A+B             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 特征工程流水线实现
# ============================================================

class FeaturePipeline:
    """
    特征工程流水线
    
    将多个特征转换器串联起来，自动执行并追踪血缘
    
    核心功能:
    1. add_transformer(): 添加转换器
    2. run(): 执行流水线，自动追踪血缘
    
    工作流程:
    ┌─────────────────────────────────────────────────────────┐
    │  输入数据                                               │
    │      ↓ 注册血缘                                        │
    │  转换器1 (normalize)                                   │
    │      ↓ 记录转换                                        │
    │  转换器2 (clip)                                        │
    │      ↓ 记录转换                                        │
    │  输出数据                                               │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, feature_store: FeatureStore, lineage_tracker: DataLineageTracker):
        """
        参数:
            feature_store: 特征存储实例
            lineage_tracker: 血缘追踪器实例
        """
        self.store = feature_store
        self.tracker = lineage_tracker
        self.transformers = []  # 转换器列表
    
    def add_transformer(self, name: str, fn, params: Dict = None):
        """
        添加转换器
        
        参数:
            name: 转换器名称
            fn: 转换函数，签名 fn(data, **params) -> transformed_data
            params: 转换参数
        """
        self.transformers.append({
            'name': name,
            'fn': fn,
            'params': params or {}
        })
    
    def run(self, input_data, input_id: str):
        """
        执行流水线
        
        参数:
            input_data: 输入数据
            input_id: 输入数据集ID
            
        返回:
            (processed_data, final_id): 处理后的数据和最终数据集ID
        """
        # 注册输入数据
        self.tracker.register_dataset(input_id, 'input', input_data)
        
        current_data = input_data
        current_id = input_id
        
        # 依次执行转换器
        for t in self.transformers:
            output_id = f"{current_id}_{t['name']}"
            
            # 执行转换
            current_data = t['fn'](current_data, **t['params'])
            
            # 记录血缘
            self.tracker.record_transformation(
                current_id, t['name'], t['params'], output_id
            )
            
            current_id = output_id
        
        return current_data, current_id


# ============================================================
# 常用特征转换函数
# ============================================================

def normalize(data, method: str = 'zscore'):
    """
    标准化转换
    
    参数:
        data: 输入数据
        method: 标准化方法
            - 'zscore': Z-score 标准化 (x - mean) / std
            - 'minmax': Min-Max 标准化 (x - min) / (max - min)
    """
    if method == 'zscore':
        return (data - np.mean(data, axis=0)) / (np.std(data, axis=0) + 1e-8)
    elif method == 'minmax':
        return (data - np.min(data, axis=0)) / (np.max(data, axis=0) - np.min(data, axis=0) + 1e-8)
    return data


def clip_outliers(data, threshold: float = 3):
    """
    异常值裁剪
    
    将超出阈值范围的值裁剪到边界
    
    参数:
        data: 输入数据
        threshold: 裁剪阈值（标准差倍数）
    """
    return np.clip(data, -threshold, threshold)


# ============================================================
# 特征流水线演示
# ============================================================
print("=" * 60)
print("特征工程流水线演示")
print("=" * 60)

# 创建流水线
pipeline = FeaturePipeline(store, tracker)

# 添加转换器
pipeline.add_transformer('normalize', normalize, {'method': 'zscore'})
pipeline.add_transformer('clip', clip_outliers, {'threshold': 2})

print("\n流水线配置:")
print("-" * 40)
for i, t in enumerate(pipeline.transformers):
    print(f"  {i+1}. {t['name']}: {t['params']}")

# 执行流水线
print("\n执行流水线:")
print("-" * 40)
raw = np.random.randn(50, 5) * 10  # 模拟原始数据（较大方差）
processed, final_id = pipeline.run(raw, 'batch_001')

print(f"  输入形状: {raw.shape}")
print(f"  输出形状: {processed.shape}")
print(f"  输入均值: {np.mean(raw):.2f}, 标准差: {np.std(raw):.2f}")
print(f"  输出均值: {np.mean(processed):.2f}, 标准差: {np.std(processed):.2f}")

# 查看血缘
print("\n数据血缘链:")
print("-" * 40)
lineage = tracker.get_full_lineage(final_id)
for record in lineage:
    print(f"  {record.dataset_id} (来源: {record.source})")

## 总结

本教程介绍了 Feature Store 和数据版本控制的核心概念：

### 核心组件回顾

| 组件 | 功能 | 关键方法 |
|:-----|:-----|:---------|
| Feature | 特征定义 | name, dtype, entity |
| FeatureStore | 特征存储 | ingest(), get_online_features(), get_historical_features() |
| DataLineage | 血缘记录 | dataset_id, parent_datasets, transformations |
| DataLineageTracker | 血缘追踪 | register_dataset(), record_transformation(), get_full_lineage() |
| FeaturePipeline | 特征流水线 | add_transformer(), run() |

### 在线 vs 离线存储对比

| 特性 | 在线存储 | 离线存储 |
|:-----|:---------|:---------|
| 延迟 | <10ms | 秒~分钟级 |
| 数据量 | 最新值 | 历史全量 |
| 用途 | 实时推理 | 模型训练 |
| 存储成本 | 高 | 低 |

### 最佳实践

```
Feature Store 检查清单:
✓ 统一训练和推理的特征计算逻辑
✓ 为每个特征定义清晰的元数据
✓ 追踪特征的完整血缘链
✓ 使用流水线自动化特征工程
✓ 定期验证特征数据质量
✓ 监控特征漂移

常见陷阱:
✗ 训练和推理使用不同的特征计算逻辑
✗ 没有记录特征的来源和转换历史
✗ 重复计算相同的特征
✗ 忽略特征的时效性（TTL）
```

### 企业级工具推荐

| 工具 | 特点 | 适用场景 |
|:-----|:-----|:---------|
| Feast | 开源、轻量 | 中小规模 |
| Tecton | 企业级、全托管 | 大规模生产 |
| Databricks Feature Store | 与 Spark 集成 | 大数据场景 |
| AWS SageMaker Feature Store | AWS 原生 | AWS 用户 |

---

## 恭喜完成 MLOps 全部教程！

您已学习了 MLOps 的核心组件：

1. ✅ **实验追踪** - 记录训练过程
2. ✅ **模型注册** - 版本管理
3. ✅ **生产监控** - 实时监控
4. ✅ **漂移检测** - 发现问题
5. ✅ **A/B 测试** - 科学对比
6. ✅ **自动重训练** - 自动修复
7. ✅ **特征存储** - 特征复用